# 📊 Diana — Monthly Data Monitor

> **VI:** Chạy mỗi tháng sau khi append data mới để phát hiện bất thường trước khi gửi báo cáo.  
> **EN:** Run each month after appending new data to catch anomalies before delivering the report.
>
> ⚙️ **Only update 3 lines in the CONFIG cell below.**

In [ ]:
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

# ── CONFIG: update every month ─────────────────────────────
REPORT_PATH   = r""
NEW_MONTH     = "05/2026"   # <-- newly appended month (MM/YYYY)
PREV_MONTH    = "04/2026"   # <-- previous month for comparison

# Alert thresholds — adjust based on business seasonality
DRIFT_WARN     = 0.30   # +/-30% MoM -> WARNING
DRIFT_ALERT    = 0.60   # +/-60% MoM -> ALERT (investigate immediately)
UNDEFINED_WARN = 0.15   # Undefined share > 15% of total market -> concern
MIN_REV_B      = 0.1    # Minimum 100M revenue to be considered significant
# ──────────────────────────────────────────────────────────

FUZZY_BRANDS = ['Undefined', 'No Brand', 'Unknown', 'Other', '']

df      = pd.read_excel(REPORT_PATH, sheet_name='Full Data')
df_new  = df[df['Time'] == NEW_MONTH].copy()
df_prev = df[df['Time'] == PREV_MONTH].copy()

total_new  = df_new['Sales Value'].sum()
total_prev = df_prev['Sales Value'].sum()

print(f"✅ Loaded: {len(df):,} total rows  |  {NEW_MONTH}: {len(df_new):,} rows  |  {PREV_MONTH}: {len(df_prev):,} rows")

✅ Loaded: 19,710 total rows  |  05/2026: 588 rows  |  04/2026: 568 rows


---
## 🔴 CHECK 1 — Undefined / No Brand spike

| VI | EN |
|----|----|
| Undefined tăng mạnh → có brand mới chưa được tagging, hoặc data source đổi cấu trúc | Spike in Undefined share → new brands not yet tagged, or upstream data structure changed |

In [2]:
undef_new  = df_new[df_new['Brand'].isin(FUZZY_BRANDS)]['Sales Value'].sum()
undef_prev = df_prev[df_prev['Brand'].isin(FUZZY_BRANDS)]['Sales Value'].sum()

undef_share_new  = undef_new  / total_new  if total_new  else 0
undef_share_prev = undef_prev / total_prev if total_prev else 0
undef_mom_change = (undef_new - undef_prev) / undef_prev if undef_prev else float('inf')

print(f"{'':=<60}")
print(f"  {PREV_MONTH}  →  Undefined: {undef_prev/1e9:,.1f}B  ({undef_share_prev:.1%} of market)")
print(f"  {NEW_MONTH}  →  Undefined: {undef_new/1e9:,.1f}B  ({undef_share_new:.1%} of market)")
print(f"  MoM change: {undef_mom_change:+.1%}")
print(f"{'':=<60}")

if undef_share_new > UNDEFINED_WARN:
    print(f"\n🔴 ALERT: Undefined = {undef_share_new:.1%} of market  >  threshold {UNDEFINED_WARN:.0%}")
    print(f"   → VI: Có thể có brand mới chưa được tagging. Cần review brand list!")
    print(f"   → EN: Likely new brands not yet tagged. Review brand mapping list!")
elif undef_mom_change > DRIFT_WARN:
    print(f"\n🟡 WARNING: Undefined +{undef_mom_change:.1%} MoM — check for new untagged brands")
else:
    print(f"\n✅ OK: Undefined is stable")

  04/2026  →  Undefined: 23.9B  (23.6% of market)
  05/2026  →  Undefined: 33.9B  (28.8% of market)
  MoM change: +41.7%

🔴 ALERT: Undefined = 28.8% of market  >  threshold 15%
   → VI: Có thể có brand mới chưa được tagging. Cần review brand list!
   → EN: Likely new brands not yet tagged. Review brand mapping list!


---
## 🆕 CHECK 2 — New brand debut

| VI | EN |
|----|----|
| Brand xuất hiện lần đầu → competitor mới vào thị trường, hoặc lỗi tagging | Brand appears for the first time → new market entrant, or tagging artifact |

In [3]:
brands_historical = set(df[df['Time'] != NEW_MONTH]['Brand'].unique())
brands_new_month  = set(df_new['Brand'].unique())
brand_debut       = brands_new_month - brands_historical

if brand_debut:
    debut_rev = (
        df_new[df_new['Brand'].isin(brand_debut)]
        .groupby('Brand')['Sales Value'].sum()
        .sort_values(ascending=False)
        .reset_index()
    )
    debut_rev['Revenue (B)']   = (debut_rev['Sales Value'] / 1e9).round(2)
    debut_rev['Market Share']  = (debut_rev['Sales Value'] / total_new).map('{:.2%}'.format)
    print(f"🆕 {len(brand_debut)} NEW BRAND(S) APPEARED IN {NEW_MONTH}:")
    print(debut_rev[['Brand', 'Revenue (B)', 'Market Share']].to_string(index=False))
    print("\n→ Action: Confirm if real new entrant vs. tagging error before sending report")
else:
    print(f"✅ No new brand debut in {NEW_MONTH}")

✅ No new brand debut in 05/2026


---
## 🟡 CHECK 3 — In-scope brand drop-off / spike

| VI | EN |
|----|----|
| Brand giảm mạnh → cảnh báo trước khi client hỏi | Brand drops sharply → flag before client raises it in review meeting |

In [4]:
rev_by_brand = (
    df[df['Time'].isin([PREV_MONTH, NEW_MONTH])]
    .groupby(['Brand', 'Time'])['Sales Value']
    .sum()
    .unstack(fill_value=0)
    .reset_index()
)

for col in [PREV_MONTH, NEW_MONTH]:
    if col not in rev_by_brand.columns:
        rev_by_brand[col] = 0

rev_by_brand['MoM']        = (rev_by_brand[NEW_MONTH] - rev_by_brand[PREV_MONTH]) / rev_by_brand[PREV_MONTH].replace(0, float('nan'))
rev_by_brand['Prev (B)']   = (rev_by_brand[PREV_MONTH] / 1e9).round(2)
rev_by_brand['New (B)']    = (rev_by_brand[NEW_MONTH]  / 1e9).round(2)

significant = rev_by_brand[
    (~rev_by_brand['Brand'].isin(FUZZY_BRANDS)) &
    (rev_by_brand[PREV_MONTH] / 1e9 >= MIN_REV_B)
].copy()

alerts = significant[significant['MoM'] < -DRIFT_ALERT].sort_values('MoM')
warns  = significant[(significant['MoM'] < -DRIFT_WARN) & (significant['MoM'] >= -DRIFT_ALERT)].sort_values('MoM')
spikes = significant[significant['MoM'] >  DRIFT_ALERT].sort_values('MoM', ascending=False)

cols_show = ['Brand', 'Prev (B)', 'New (B)', 'MoM']

def fmt_mom(df):
    out = df[cols_show].copy()
    out['MoM'] = out['MoM'].map('{:+.1%}'.format)
    return out

if not alerts.empty:
    print(f"🔴 ALERT — Dropped > {DRIFT_ALERT:.0%} MoM  (missing data or real decline?):")
    print(fmt_mom(alerts).to_string(index=False)); print()

if not warns.empty:
    print(f"🟡 WARNING — Dropped {DRIFT_WARN:.0%}–{DRIFT_ALERT:.0%} MoM:")
    print(fmt_mom(warns).to_string(index=False)); print()

if not spikes.empty:
    print(f"⚡ SPIKE — Surged > {DRIFT_ALERT:.0%} MoM  (verify prev month wasn't missing data):")
    print(fmt_mom(spikes).to_string(index=False)); print()

if alerts.empty and warns.empty and spikes.empty:
    print("✅ All in-scope brands within acceptable MoM range")

🟡 WARNING — Dropped 30%–60% MoM:
Brand  Prev (B)  New (B)    MoM
Kazuo       0.2     0.12 -36.7%



---
## 📋 CHECK 4 — Market overview MoM

In [5]:
mom_total = (total_new - total_prev) / total_prev if total_prev else float('inf')

print(f"Total market revenue:")
print(f"  {PREV_MONTH}: {total_prev/1e9:,.1f}B")
print(f"  {NEW_MONTH}:  {total_new/1e9:,.1f}B   ({mom_total:+.1%} MoM)")
print()

plat = (
    df[df['Time'].isin([PREV_MONTH, NEW_MONTH])]
    .groupby(['Platform', 'Time'])['Sales Value'].sum()
    .unstack(fill_value=0)
)
for col in [PREV_MONTH, NEW_MONTH]:
    if col not in plat.columns: plat[col] = 0

plat['MoM']       = ((plat[NEW_MONTH] - plat[PREV_MONTH]) / plat[PREV_MONTH].replace(0, float('nan'))).map('{:+.1%}'.format)
plat['Prev (B)']  = (plat[PREV_MONTH] / 1e9).round(1)
plat['New (B)']   = (plat[NEW_MONTH]  / 1e9).round(1)

print("Revenue by Platform:")
print(plat[['Prev (B)', 'New (B)', 'MoM']].to_string())

Total market revenue:
  04/2026: 101.6B
  05/2026:  117.7B   (+15.9% MoM)

Revenue by Platform:
Time      Prev (B)  New (B)     MoM
Platform                           
Lazada         0.5      0.3  -27.7%
Shopee        48.8     47.4   -2.8%
TikTok        52.4     70.0  +33.6%


---
## ✅ SUMMARY — Go / No-go before sending report

In [6]:
issues = []

if undef_share_new > UNDEFINED_WARN:
    issues.append(f"🔴 Undefined = {undef_share_new:.1%} of market — review brand tagging")
elif undef_mom_change > DRIFT_WARN:
    issues.append(f"🟡 Undefined +{undef_mom_change:.1%} MoM — monitor next month")

if brand_debut:
    issues.append(f"🆕 {len(brand_debut)} new brand(s): {', '.join(sorted(brand_debut))} — confirm before publish")

if not alerts.empty:
    issues.append(f"🔴 {len(alerts)} brand(s) dropped >60% MoM: {', '.join(alerts['Brand'].tolist())}")

if not warns.empty:
    issues.append(f"🟡 {len(warns)} brand(s) dropped 30–60% MoM: {', '.join(warns['Brand'].tolist())}")

if not spikes.empty:
    issues.append(f"⚡ {len(spikes)} brand(s) spiked >60% MoM: {', '.join(spikes['Brand'].tolist())}")

print(f"{'':=<65}")
print(f"  DIANA DATA MONITOR — {NEW_MONTH}")
print(f"{'':=<65}")
if issues:
    print(f"  ⚠️  {len(issues)} issue(s) to review before sending:")
    for i, issue in enumerate(issues, 1):
        print(f"  {i}. {issue}")
    print(f"\n  → VI: Cần xử lý các mục trên trước khi gửi báo cáo")
else:
    print(f"  ✅ Data looks stable — OK to send report")
    print(f"  ✅ Dữ liệu ổn định — OK để gửi báo cáo")
print(f"{'':=<65}")

  DIANA DATA MONITOR — 05/2026
  ⚠️  2 issue(s) to review before sending:
  1. 🔴 Undefined = 28.8% of market — review brand tagging
  2. 🟡 1 brand(s) dropped 30–60% MoM: Kazuo

  → VI: Cần xử lý các mục trên trước khi gửi báo cáo
